# Distribuição de proprietários 

**Pergunta de pesquisa:** A distribuição de proprietários estimados segue uma "long tail"? Ou seja, poucos jogos concentram a maioria dos jogadores?

**Responsável:** João Pedro Alves Menicucci

## Configuração

Funciona localmente e no Google Colab.

In [ ]:
import os, sys
from pathlib import Path

if "google.colab" in sys.modules:
    !git clone -q https://github.com/pbarruetavena/tp_ciencia_de_dados.git
    %cd tp_ciencia_de_dados
    !pip install -q kagglehub
    !python -m src.download_data
    sys.path.insert(0, os.getcwd())
else:
    # Permite importar src/ a partir de notebooks/<pasta>/
    sys.path.insert(0, str(Path.cwd().parents[1]))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.config import FIGURAS
from src.dados import carregar_jogos
from src.limpeza import limpar

jogos = limpar(carregar_jogos())
x = jogos["owners_mid"].dropna()
x.shape

## Análise exploratória

O atributo `estimated_owners` vem em faixas (ex.: `"0 - 20000"`); `src.limpeza.limpar()` já converte cada faixa para o ponto médio numérico `owners_mid` e trata `"0 - 0"` como NaN (falta de dado do SteamSpy). Sobram 13 faixas distintas, e a mais baixa (`0 - 20000`) concentra cerca de 80% dos jogos — a base é grosseiramente agregada.

**O que se espera encontrar:** se a distribuição for uma *long tail*, a média fica muito acima da mediana e poucos jogos (os mega-hits) concentram a maioria dos jogadores.

**Como verificar (abordagem IQR):** calculamos os quartis de `owners_mid`, definimos o limite superior de outliers `LS = Q3 + 1.5*IQR` e medimos que fração do total de jogadores está acima dele. Com a maior parte da massa na primeira faixa, esperamos `Q1 = Q3` (IQR = 0) na amostra completa — por isso repetimos o mesmo cálculo na cauda (jogos acima da mediana), onde o IQR é informativo.

In [ ]:
media, dp = x.mean(), x.std(ddof=1)
q1, q2, q3 = x.quantile([0.25, 0.5, 0.75])
iqr = q3 - q1
ls = q3 + 1.5 * iqr

print(f"n = {len(x):,}   média = {media:,.0f}   dp = {dp:,.0f}")
print(f"Q1 = {q1:,.0f}   Q2 (mediana) = {q2:,.0f}   Q3 = {q3:,.0f}")
print(f"IQR = {iqr:,.0f}   LS = Q3 + 1.5*IQR = {ls:,.0f}")

In [ ]:
outliers = x > ls
n_out = int(outliers.sum())
conc_out = x[outliers].sum() / x.sum() * 100

print(f"Outliers superiores (x > LS): {n_out:,} jogos ({outliers.mean() * 100:.1f}%)")
print(f"Concentração: {conc_out:.1f}% de todos os jogadores estimados estão nos outliers")

In [ ]:
cauda = x[x > q2]
q1c, q3c = cauda.quantile([0.25, 0.75])
iqr_cauda = q3c - q1c
ls_cauda = q3c + 1.5 * iqr_cauda

mega = x > ls_cauda
print(f"Cauda (x > {q2:,.0f}): n = {len(cauda):,}")
print(f"Q1 = {q1c:,.0f}   Q3 = {q3c:,.0f}   IQR = {iqr_cauda:,.0f}   LS = {ls_cauda:,.0f}")
print(f"Mega-hits (x > LS da cauda): {int(mega.sum()):,} jogos ({mega.mean() * 100:.1f}%) "
      f"concentram {x[mega].sum() / x.sum() * 100:.1f}% dos jogadores")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(y=x, ax=ax1)
ax1.set_yscale("log")
ax1.set(title="Amostra completa (caixa colapsa: IQR = 0)",
        ylabel="Proprietários (escala log)")

sns.boxplot(y=cauda, ax=ax2)
ax2.set_yscale("log")
ax2.set(title=f"Cauda (x > {q2:,.0f})", ylabel="Proprietários (escala log)")

fig.savefig(FIGURAS / "boxplot_owners.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(x, bins=np.logspace(4, np.log10(x.max()), 40), ax=ax)
ax.set_xscale("log")
ax.set(title="Histograma de proprietários estimados",
       xlabel="Proprietários (escala log)", ylabel="Jogos")
fig.savefig(FIGURAS / "histograma_owners.png", dpi=150, bbox_inches="tight")
plt.show()

**Interpretação dos gráficos:** no boxplot da amostra completa a caixa colapsa na mediana (IQR = 0) mesmo em escala log — os outliers, todos acima da primeira faixa, vão até 150 milhões de proprietários; o boxplot da cauda mostra a caixa de forma legível (Q1 = 35.000, Q3 = 150.000) e a cauda acima dela. O histograma (eixo x em log) cai bruscamente já na primeira faixa e arrasta uma cauda longa à direita.

**Leitura dos resultados:**

- **Visão A (amostra completa):** o IQR dá 0 porque ~80% dos jogos está na faixa `0 - 20000` (Q1 = Q3 = mediana = 10.000), efeito da granularidade das faixas do SteamSpy — que é, em si, evidência de concentração. Com `LS = Q3 + 1.5*IQR = 10.000`, os outliers superiores são ~1/5 dos jogos e concentram ~91% de todos os jogadores estimados.
- **Visão B (cauda, acima da mediana):** aqui o IQR é informativo (Q1 = 35.000, Q3 = 150.000) e `LS = 322.500` separa os mega-hits: ~4,4% dos jogos concentram sozinhos ~79% dos jogadores.

Em qualquer uma das duas visões, poucos jogos concentram a maioria dos jogadores — o padrão de *long tail* que a pergunta de pesquisa previa.

## Teste de hipótese e intervalo de confiança

### Intervalo de confiança de 95% para a média

Estimamos o IC por **bootstrap**: reamostramos os jogos com reposição B = 10.000 vezes, calculamos a média em cada reamostragem e tomamos os percentis 2,5% e 97,5% dessa distribuição bootstrap.

In [ ]:
B = 10_000
boot = stats.bootstrap(
    (x.to_numpy(),),
    np.mean,
    n_resamples=B,
    batch=200,
    method="percentile",
    confidence_level=0.95,
    random_state=np.random.default_rng(42),
)
ic95 = boot.confidence_interval

print(f"Média = {media:,.0f}   dp = {dp:,.0f}   n = {len(x):,}   B = {B:,} reamostragens")
print(f"IC 95% (bootstrap, percentis): [{ic95.low:,.0f}, {ic95.high:,.0f}]")
print(f"Mediana = {q2:,.0f} -> "
      f"{'abaixo do limite inferior' if q2 < ic95.low else 'dentro do intervalo'}")

**Comparação média vs. mediana:** a mediana (10.000 proprietários, o jogo "típico") fica muito abaixo do limite inferior do IC da média (≈ 80 mil). A média é puxada pelos mega-hits e não representa o jogo típico — evidência estatística robusta de uma distribuição fortemente assimétrica à direita.

### Teste de aderência — Kolmogorov-Smirnov

Testamos duas candidatas para a forma da distribuição de proprietários:

- **H₀¹:** a distribuição de proprietários estimados é Exponencial.
- **H₀²:** a distribuição de proprietários estimados é Log-normal.
- **H₁:** a distribuição não segue a candidata testada.
- **Teste:** Kolmogorov-Smirnov de uma amostra (`scipy.stats.kstest`) para cada candidata, com parâmetros estimados dos próprios dados (λ = 1/média na Exponencial; μ e σ de `ln(owners_mid)` na Log-normal).
- **Nível de significância:** α = 0,05

In [ ]:
lam = 1 / media
estatistica, p_valor = stats.kstest(x, "expon", args=(0, media))

print(f"lambda = 1 / média = {lam:.3e}")
print(f"Estatística K-S = {estatistica:.4f}   p-valor = {p_valor:.3e}")

In [ ]:
logx = np.log(x)
mu_ln, sigma_ln = logx.mean(), logx.std(ddof=1)
estat_ln, p_ln = stats.kstest(x, "lognorm", args=(sigma_ln, 0, np.exp(mu_ln)))

print(f"mu = {mu_ln:.4f}   sigma = {sigma_ln:.4f}")
print(f"Estatística K-S = {estat_ln:.4f}   p-valor = {p_ln:.3e}")

In [ ]:
t = np.logspace(4, np.log10(x.max()), 400)

fig, ax = plt.subplots(figsize=(8, 5))
sns.ecdfplot(x=x, ax=ax, label="Empírica")
ax.plot(t, stats.expon.cdf(t, scale=media), label="Exponencial")
ax.plot(t, stats.lognorm.cdf(t, sigma_ln, scale=np.exp(mu_ln)), label="Log-normal")
ax.set_xscale("log")
ax.set(title="CDFs teóricas x CDF empírica de proprietários",
       xlabel="Proprietários estimados (escala log)", ylabel="F(x)")
ax.legend()
fig.savefig(FIGURAS / "cdf_owners.png", dpi=150, bbox_inches="tight")
plt.show()

**Resultado:** os dois p-valores são praticamente zero, muito menores que α = 0,05. Rejeitamos H₀¹ e H₀².

**Comparação:** a estatística K-S da Log-normal (≈ 0,46) é bem menor que a da Exponencial (≈ 0,68) — a log-normal descreve melhor a cauda pesada, mas ainda assim não adere.

**Interpretação:** a anatomia dos dados (acumulados em intervalos) dificulta a análise sob Kolmogorov-Smirnov.

### Teste de aderência — Qui-quadrado (comparação com a Exponencial e a Log-normal)

O K-S compara a CDF empírica com a teórica; o qui-quadrado de aderência compara contagens observadas e esperadas faixa a faixa — mais natural aqui, já que a base só vem em faixas. Testamos as mesmas duas candidatas do K-S:

- **H₀¹:** a distribuição de proprietários estimados é Exponencial (com λ = 1/média).
- **H₀²:** a distribuição de proprietários estimados é Log-normal (com μ e σ estimados de `ln(owners_mid)`).
- **H₁:** a distribuição não segue a candidata testada.
- **Teste:** qui-quadrado de aderência de uma amostra (`scipy.stats.chisquare`) para cada candidata, agrupando os jogos nas faixas da Steam e comparando com a probabilidade teórica em cada faixa. Como os parâmetros são estimados dos dados, descontamos um grau de liberdade por parâmetro (`ddof=1` na Exponencial, `ddof=2` na Log-normal).
- **Nível de significância:** α = 0,05

**Por que ele é adequado:** trabalhamos com contagens por faixa, mantendo frequência esperada ≥ 5 em todas elas (a cauda foi agrupada a partir de 500.000 justamente por isso).

In [ ]:
bordas = np.array([0, 20_000, 50_000, 100_000, 200_000, 500_000, np.inf])
observado = pd.cut(x, bins=bordas).value_counts().sort_index()
esperado_exp = len(x) * np.diff(stats.expon.cdf(bordas, scale=media))
esperado_ln = len(x) * np.diff(stats.lognorm.cdf(bordas, sigma_ln, scale=np.exp(mu_ln)))

quadro = pd.DataFrame({
    "faixa": observado.index.astype(str),
    "observado": observado.to_numpy(),
    "esperado_exp": esperado_exp,
    "esperado_ln": esperado_ln,
})
print(quadro.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print(f"\nmenor frequência esperada = {min(esperado_exp.min(), esperado_ln.min()):,.0f}")

estat_q_exp, p_q_exp = stats.chisquare(observado.to_numpy(), esperado_exp, ddof=1)
estat_q_ln, p_q_ln = stats.chisquare(observado.to_numpy(), esperado_ln, ddof=2)
print(f"Exponencial: qui-quadrado = {estat_q_exp:,.1f}   gl = {len(observado) - 2}   p-valor = {p_q_exp:.3e}")
print(f"Log-normal:  qui-quadrado = {estat_q_ln:,.1f}   gl = {len(observado) - 3}   p-valor = {p_q_ln:.3e}")

**Resultado:** os dois p-valores são praticamente zero, muito menores que α = 0,05. Rejeitamos H₀¹ e H₀² — em acordo com o K-S.

**Comparação:** a Log-normal tem qui-quadrado bem menor que a Exponencial (≈ 110 mil contra ≈ 276 mil), descrevendo melhor a distribuição, mas ainda assim não adere.

**Interpretação:** a tabela mostra onde cada candidata falha. A Exponencial prevê ~25 mil jogos na primeira faixa, onde há ~97 mil, e despeja massa demais nas intermediárias (prevê ~30 mil entre 50 mil e 100 mil, contra ~5 mil observados). A Log-normal chega mais perto (~72 mil na primeira faixa), mas ainda subestima a concentração inicial, exagera nas faixas intermediárias e subestima a cauda de mega-hits (prevê ~75 jogos acima de 500 mil, contra ~2,5 mil observados). O decaimento real é muito mais brusco no início e a cauda é mais pesada do que qualquer uma das duas.

## Resultado

**Resultado:** ~21% dos jogos (outliers superiores acima de LS = 10.000) concentram ~91% de todos os jogadores estimados; recortando a cauda, ~4,4% dos mega-hits (acima de LS = 322.500) concentram ~79%. A mediana (10.000) fica muito abaixo do IC 95% da média por bootstrap (≈ [80 mil; 95 mil]) e o teste K-S rejeita tanto a Exponencial quanto a Log-normal (p ≈ 0; a log-normal chega mais perto, D ≈ 0,46 contra 0,68), assim como o qui-quadrado de aderência, contra a Exponencial (χ² ≈ 276 mil) e contra a Log-normal (χ² ≈ 110 mil), ambos com p ≈ 0.

**Conclusão:** sim — a distribuição de proprietários estimados é uma *long tail* extrema: poucos jogos concentram a maioria dos jogadores, e nenhuma das duas distribuições clássicas de cauda captura essa concentração.